In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Verification Gate
input_file = "master_feature_table_with_hazards.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Missing master database file! Please run Notebook 06 first.")

In [3]:
master = pd.read_csv(input_file)
print("=====================================================================")
print("📊 STRESS TESTING LABORATORY: MASTER FEATURE DATASET INGESTED")
print("=====================================================================")
print(f"Verified Matrix Rows: {master.shape[0]} (Expected: 18260 rows)")
print(f"Verified Matrix Columns: {master.shape[1]}")

📊 STRESS TESTING LABORATORY: MASTER FEATURE DATASET INGESTED
Verified Matrix Rows: 18260 (Expected: 18260 rows)
Verified Matrix Columns: 67


## : Build Predefined Scenario Library

In [4]:
# Reusable laboratory experiments representing progressive severity scales
SCENARIOS = {
    "heavy_rain": {
        "rain": 50.0
    },
    "extreme_rain": {
        "rain": 100.0
    },
    "cloudburst": {
        "rain": 200.0
    },
    "tourist_peak": {
        "crowd_baseline": 90.0
    },
    "worst_case": {
        "rain": 120.0,
        "wind_speed": 60.0,
        "festival_boost": 15.0,
        "crowd_baseline": 90.0
    }
}

print("Predefined experimental stress scenarios successfully loaded into memory.")

Predefined experimental stress scenarios successfully loaded into memory.


## Scenario Injection Engine

In [5]:
def recompute_hazard_matrix(row):
    """
    Re-runs the underlying hazard calculations, implementing a smooth rainfall saturation
    gradient for landslides and active multi-channel infrastructure cascades for transport.
    """
    # -----------------------------------------------------------------
    # 1. WEATHER HAZARD SUB-SCORE
    # -----------------------------------------------------------------
    rain_p = min((row['rain'] / 50.0) * 50.0, 50.0)
    snow_val = row.get('snowfall', 0.0)
    snow_p = min((snow_val / 20.0) * 30.0, 30.0)
    wind_p = min((row['wind_speed'] / 40.0) * 20.0, 20.0)
    weather_hazard = min(rain_p + snow_p + wind_p, 100.0)
    
    # -----------------------------------------------------------------
    # 2. LANDSLIDE HAZARD SUB-SCORE (Smooth Rain Saturation Upgrade)
    # -----------------------------------------------------------------
    distance_factor = 100.0 / (row['nearest_landslide_km'] + 1.0)
    density_factor = row['landslide_density_per_1000sqkm'] * 50.0
    terrain_factor = row['mountain_flag'] * 20.0
    base_landslide = (distance_factor * 0.4) + (density_factor * 0.3) + (terrain_factor * 0.3)
    
    # UPGRADE 1: Smooth continuous rainfall saturation multiplier
    rain_multiplier = 1.0 + min(row['rain'] / 200.0, 0.8)
    if row['mountain_flag'] == 1:
        rain_multiplier *= 1.3
        
    landslide_hazard = min(base_landslide * rain_multiplier, 100.0)
    
    # -----------------------------------------------------------------
    # 3. CROWD HAZARD SUB-SCORE
    # -----------------------------------------------------------------
    crowd_hazard = min(row['crowd_baseline'] + row['festival_boost'] + (row['school_vacation_flag'] * 10.0), 100.0)
    
    # -----------------------------------------------------------------
    # 4. TRANSPORT HAZARD SUB-SCORE (Cascading Road Closure Upgrade)
    # -----------------------------------------------------------------
    base_transport = (
        (row['transport_complexity_score'] * 0.5) + 
        (row['budget_stress_index'] * 0.3) + 
        (row['elevation_penalty'] * 0.2)
    )
    transport_hazard = base_transport
    
    # UPGRADE 2: Extreme rain cascades into road networks causing emergency evacuation delay
    if row['rain'] > 80.0:
        transport_hazard *= 1.20
        if row['mountain_flag'] == 1:
            transport_hazard *= 1.15
            
    transport_hazard = min(transport_hazard, 100.0)
    
    # -----------------------------------------------------------------
    # 5. OVERALL COMPOSITE WEIGHTED BLEND RISK
    # -----------------------------------------------------------------
    overall_hazard = (weather_hazard * 0.35) + (landslide_hazard * 0.25) + (crowd_hazard * 0.20) + (transport_hazard * 0.20)
    
    return weather_hazard, landslide_hazard, crowd_hazard, transport_hazard, overall_hazard

In [6]:
def simulate_scenario(location, date, scenario_name):
    """
    Clones a target timeline observation record, injects predefined stressors,
    and returns a side-by-side comparison matrix.
    """
    match = master[(master['location'] == location) & (master['date'] == date)]
    if match.empty:
        raise ValueError(f"No match found for location '{location}' on date '{date}'")
        
    base_row = match.iloc[0].copy()
    scenario_row = base_row.copy()
    
    # Apply experiment overrides
    overrides = SCENARIOS[scenario_name]
    for key, val in overrides.items():
        scenario_row[key] = val
        
    # Recalculate states
    wb, lb, cb, tb, ob = base_row['weather_hazard_score'], base_row['landslide_hazard_score'], base_row['crowd_hazard_score'], base_row['transport_hazard_score'], base_row['overall_hazard_score']
    ws, ls, cs, ts, os_val = recompute_hazard_matrix(scenario_row)
    
    return {
        "location": location, "date": date, "scenario_name": scenario_name,
        "base_w": wb, "base_l": lb, "base_c": cb, "base_t": tb, "base_risk": ob,
        "scen_w": ws, "scen_l": ls, "scen_c": cs, "scen_t": ts, "scenario_risk": os_val
    }

print("Upgraded Simulation Engine with continuous saturation math and transit cascades compiled.")

Upgraded Simulation Engine with continuous saturation math and transit cascades compiled.


## Delta Calculator

In [7]:
# Run an experiment testing extreme rain on a baseline winter day in Manali
exp_results = simulate_scenario(location="Manali", date="2021-01-01", scenario_name="extreme_rain")

# Process risk shifts
risk_change = exp_results['scenario_risk'] - exp_results['base_risk']
percent_change = (risk_change / exp_results['base_risk']) * 100.0

In [8]:
print("=====================================================================")
print("📉 INDIVIDUAL OBSERVATION RISK DELTA REPORT")
print("=====================================================================")
print(f"Target Location:  {exp_results['location']} ({exp_results['date']})")
print(f"Experiment Run:   {exp_results['scenario_name']}")
print(f" -> Base Risk Index:  {exp_results['base_risk']:.2f}")
print(f" -> Scenario Risk:    {exp_results['scenario_risk']:.2f}")
print(f" -> Absolute Delta:   +{risk_change:.2f} risk points")
print(f" -> Percentage Shift: +{percent_change:.2f}%")

📉 INDIVIDUAL OBSERVATION RISK DELTA REPORT
Target Location:  Manali (2021-01-01)
Experiment Run:   extreme_rain
 -> Base Risk Index:  20.92
 -> Scenario Risk:    48.10
 -> Absolute Delta:   +27.18 risk points
 -> Percentage Shift: +129.88%


## Vulnerability Analysis

In [9]:
# Isolate component jumps
weather_delta = exp_results['scen_w'] - exp_results['base_w']
landslide_delta = exp_results['scen_l'] - exp_results['base_l']
crowd_delta = exp_results['scen_c'] - exp_results['base_c']
transport_delta = exp_results['scen_t'] - exp_results['base_t']

delta_map = {
    "Weather": weather_delta, "Landslide": landslide_delta, 
    "Crowd": crowd_delta, "Transport": transport_delta
}

In [10]:
# Determine the absolute most sensitive channel under stress
largest_delta_driver = max(delta_map, key=delta_map.get)

print("=====================================================================")
print("🌋 RISK ATTRIBUTION ANALYSIS INTERVIEW PREVIEW")
print("=====================================================================")
for channel, d_val in delta_map.items():
    print(f" -> Sub-channel Point Delta ({channel}): +{d_val:.2f} points")
print(f"\n🏆 PRIMARY STRUCTURAL VULNERABILITY DETECTED: {largest_delta_driver}")

🌋 RISK ATTRIBUTION ANALYSIS INTERVIEW PREVIEW
 -> Sub-channel Point Delta (Weather): +50.00 points
 -> Sub-channel Point Delta (Landslide): +20.34 points
 -> Sub-channel Point Delta (Crowd): +0.00 points
 -> Sub-channel Point Delta (Transport): +22.95 points

🏆 PRIMARY STRUCTURAL VULNERABILITY DETECTED: Weather


## Global Comparative Leaderboard

In [11]:
leaderboard_records = []
fixed_test_date = "2021-01-01"

for city in master['location'].unique():
    res = simulate_scenario(location=city, date=fixed_test_date, scenario_name="extreme_rain")
    increase = res['scenario_risk'] - res['base_risk']
    
    leaderboard_records.append({
        "Location": city,
        "Base Risk": round(res['base_risk'], 2),
        "Scenario Risk": round(res['scenario_risk'], 2),
        "Risk Increase": round(increase, 2)
    })

In [12]:
# Sort leaderboard descending to surface the most exposed destinations
df_leaderboard = pd.DataFrame(leaderboard_records).sort_values(by="Risk Increase", ascending=False)

print("=====================================================================")
print("🏆 GLOBAL SCENARIO LEADERBOARD: GEOGRAPHIC REFLECTION CHECK")
print("=====================================================================")
df_leaderboard

🏆 GLOBAL SCENARIO LEADERBOARD: GEOGRAPHIC REFLECTION CHECK


,Location,Base Risk,Scenario Risk,Risk Increase
8,Munnar,25.74,56.08,30.35
5,Darjeeling,22.68,51.27,28.59
3,Nainital,21.41,48.98,27.58
4,Leh,27.65,55.08,27.43
9,Ooty,21.81,49.17,27.37
1,Shimla,21.46,48.68,27.22
0,Manali,20.92,48.10,27.18
2,Mussoorie,20.69,47.37,26.68
6,Goa,15.55,35.74,20.19
7,Jaipur,11.96,30.33,18.37


## Vulnerability Stress Ranking Index (Version Independent Fix)

In [13]:
print("=====================================================================")
print("🏔️ FINAL INFRASTRUCTURAL STRESS RANKING: TORRENTIAL RAINFALL VULNERABILITY")
print("=====================================================================")

# Explicit row key extraction removes any finicky space attribute errors
for rank, (index, row) in enumerate(df_leaderboard.iterrows(), 1):
    print(f"{rank}. {row['Location']} (Continuous Delta: +{row['Risk Increase']:.2f} risk points)")

🏔️ FINAL INFRASTRUCTURAL STRESS RANKING: TORRENTIAL RAINFALL VULNERABILITY
1. Munnar (Continuous Delta: +30.35 risk points)
2. Darjeeling (Continuous Delta: +28.59 risk points)
3. Nainital (Continuous Delta: +27.58 risk points)
4. Leh (Continuous Delta: +27.43 risk points)
5. Ooty (Continuous Delta: +27.37 risk points)
6. Shimla (Continuous Delta: +27.22 risk points)
7. Manali (Continuous Delta: +27.18 risk points)
8. Mussoorie (Continuous Delta: +26.68 risk points)
9. Goa (Continuous Delta: +20.19 risk points)
10. Jaipur (Continuous Delta: +18.37 risk points)


## Complete Systemic Combinations Export

In [14]:
simulation_archive = []
print("--- RUNNING MASS SCALE LABORATORY SYSTEM EXPERIMENT SIMULATOR ---")

# Evaluate all 18,260 days across all 5 configurations to yield 91,300 simulated states
for idx, base_row in master.iterrows():
    city_name = base_row['location']
    date_stamp = base_row['date']
    
    for scen_name, overrides in SCENARIOS.items():
        scen_row = base_row.copy()
        for k, v in overrides.items():
            scen_row[k] = v
            
        # Recompute advanced risk calculations
        wb, lb, cb, tb, ob = base_row['weather_hazard_score'], base_row['landslide_hazard_score'], base_row['crowd_hazard_score'], base_row['transport_hazard_score'], base_row['overall_hazard_score']
        ws, ls, cs, ts, os_val = recompute_hazard_matrix(scen_row)
        
        # Calculate shifts
        w_d = ws - wb
        l_d = ls - lb
        c_d = cs - cb
        t_d = ts - tb
        r_d = os_val - ob
        
        deltas = {'weather': w_d, 'landslide': l_d, 'crowd': c_d, 'transport': t_d}
        dominant_v = max(deltas, key=deltas.get) if any(v != 0 for v in deltas.values()) else "none"
        
        simulation_archive.append({
            "location": city_name,
            "date": date_stamp,
            "base_risk": round(ob, 4),
            "scenario_risk": round(os_val, 4),
            "risk_delta": round(r_d, 4),
            "weather_delta": round(w_d, 4),
            "landslide_delta": round(l_d, 4),
            "crowd_delta": round(c_d, 4),
            "transport_delta": round(t_d, 4),
            "dominant_vulnerability": dominant_v,
            "scenario_name": scen_name
        })

--- RUNNING MASS SCALE LABORATORY SYSTEM EXPERIMENT SIMULATOR ---


In [15]:
# Store output to disk
df_results = pd.DataFrame(simulation_archive)
df_results.to_csv("scenario_simulation_results.csv", index=False)

print("\n=====================================================================")
print("🎉 TASK 07A PIPELINE COMPLETE: SIMULATION ENGINE CLOSED")
print("=====================================================================")
print(f" -> Output Asset Generated: 'scenario_simulation_results.csv'")
print(f" -> Total Simulated States Successfully Logged: {len(df_results)} rows")


🎉 TASK 07A PIPELINE COMPLETE: SIMULATION ENGINE CLOSED
 -> Output Asset Generated: 'scenario_simulation_results.csv'
 -> Total Simulated States Successfully Logged: 91300 rows


In [16]:
df_results.head()

,location,date,base_risk,scenario_risk,risk_delta,weather_delta,landslide_delta,crowd_delta,transport_delta,dominant_vulnerability,scenario_name
0,Manali,2021-01-01,20.9241,41.7698,20.8457,50.00,13.3825,0.00,0.0000,weather,heavy_rain
1,Manali,2021-01-01,20.9241,48.0994,27.1753,50.00,20.3415,0.00,22.9493,weather,extreme_rain
2,Manali,2021-01-01,20.9241,50.1870,29.2629,50.00,28.6922,0.00,22.9493,weather,cloudburst
3,Manali,2021-01-01,20.9241,37.8600,16.9359,0.00,6.4236,76.65,0.0000,crowd,tourist_peak
4,Manali,2021-01-01,20.9241,72.3028,51.3787,67.65,23.1250,86.65,22.9493,crowd,worst_case
